# 01 · Seleção dos projetos do piloto

**Objetivo.** Escolher, entre os repositórios Python da base AIDev (pull requests abertos por agentes de IA), os cinco projetos que compõem o piloto.

- **Lê:** `../data/repository.parquet`, `../data/pull_request.parquet`
- **Grava:** `../out/candidatos.csv`, `../out/piloto.csv`

**O que esta célula faz.** Importa `pandas`, `statsmodels` e `lifelines` para confirmar que as bibliotecas usadas nas análises estão instaladas no ambiente do notebook.

In [1]:
import pandas, statsmodels, lifelines; 
print('ok')

ok


**O que isto mostra.** A saída `ok` indica que as três importações funcionaram. Ela não informa versões nem diz nada sobre os dados.

**O que esta célula faz.** Executa `ruff --version` no shell para registrar a versão do Ruff, o linter (analisador estático) de Python que será usado nas etapas seguintes para contar avisos de qualidade.

In [2]:
!ruff --version

ruff 0.16.8


**O que isto mostra.** O ambiente tem o Ruff na versão 0.16.8. Esse número registra a ferramenta usada nas medições posteriores e não é um resultado do estudo.

**O que esta célula faz.** Lê as tabelas de repositórios e de pull requests, mantém os PRs aceitos (`merged_at` preenchido) de repositórios cuja linguagem é Python e, por repositório, conta os PRs aceitos, registra a data do primeiro aceite e o agente mais frequente. Ficam como candidatos os repositórios com pelo menos 5 PRs aceitos, com nome e número de estrelas, gravados em `../out/candidatos.csv`.

In [3]:
import pandas as pd
repo = pd.read_parquet('../data/repository.parquet')
pr = pd.read_parquet('../data/pull_request.parquet')
aceitos = pr[pr.merged_at.notna()]
py = repo[repo.language == 'Python']
cont = aceitos[aceitos.repo_id.isin(py.id)].groupby('repo_id').agg(
    n_prs=('id', 'size'), primeiro=('merged_at', 'min'),
    agente=('agent', lambda s: s.value_counts().idxmax()))
cand = cont[cont.n_prs >= 5].join(py.set_index('id')[['full_name', 'stars']])
cand.to_csv('../out/candidatos.csv')  # caminhos relativos a notebooks/

**O que esta célula faz.** Filtra os candidatos pela lista fixa de cinco nomes do piloto, grava `../out/piloto.csv` e exibe a tabela resultante (o comentário do código indica que devem aparecer 5 linhas).

In [4]:
piloto = ['lidofinance/diffyscan', 'numbagg/numbagg', 'econ-ark/HARK',
          'NewFuture/DDNS', 'airbytehq/PyAirbyte']
sel = cand[cand.full_name.isin(piloto)]
sel.to_csv('../out/piloto.csv')
sel  # devem aparecer 5 linhas

,n_prs,primeiro,agente,full_name,stars
repo_id,,,,,
25501620,12,2025-06-17T16:01:43Z,Claude_Code,numbagg/numbagg,226
50448254,8,2025-05-20T00:59:13Z,OpenAI_Codex,econ-ark/HARK,352
60844169,13,2025-07-01T16:29:37Z,Copilot,NewFuture/DDNS,4528
631200841,6,2025-05-22T17:47:45Z,OpenAI_Codex,lidofinance/diffyscan,139
752526884,14,2025-01-28T17:49:30Z,Devin,airbytehq/PyAirbyte,281


**O que isto mostra.** Aparecem as 5 linhas esperadas, o que confirma que os cinco nomes da lista estão entre os candidatos (Python, ao menos 5 PRs aceitos). O número de PRs aceitos vai de 6 (diffyscan) a 14 (PyAirbyte), e o agente mais frequente é Claude_Code em numbagg, OpenAI_Codex em HARK e diffyscan, Copilot em DDNS e Devin em PyAirbyte. As estrelas variam de 139 (diffyscan) a 4528 (DDNS), e a data do primeiro PR aceito vai de 2025-01-28 (PyAirbyte) a 2025-07-01 (DDNS). A coluna `agente` indica apenas o agente mais frequente nos PRs do AIDev, não o único, e a seleção é de 5 projetos para um piloto.